In [1]:
!pip install absl-py
!pip install ml-collections

In [2]:
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession
from replay.session_handler import get_spark_session, State
from replay.splitters import UserSplitter
from replay.utils import convert2spark, get_log_info
from replay.logger import get_logger

from replay.models import UCB, KL_UCB, Wilson, RandomRec, PopRec, LinUCB, ALSWrap, ItemKNN, SLIM, LightFMWrap
from replay.obp_evaluation.replay_offline import RePlayOfflinePolicyLearner
from replay.obp_evaluation.utils import get_est_rewards_by_reg

from sklearn.ensemble import RandomForestClassifier as RandomForest
from sklearn.linear_model import LogisticRegression
from replay.obp_evaluation.utils import bandit_subset

import obp
from obp.dataset import (
    SyntheticBanditDataset,
    logistic_reward_function,
    linear_reward_function,
    OpenBanditDataset
)
from obp.policy import (
    IPWLearner, 
    QLearner,
    NNPolicyLearner, 
    Random
)

from obp.ope import (
    OffPolicyEvaluation,
    RegressionModel,
    DirectMethod,
    InverseProbabilityWeighting,
    DoublyRobust
)

In [3]:
spark = State(get_spark_session()).session
spark.sparkContext.setLogLevel('ERROR')

23/07/19 20:14:07 WARN Utils: Your hostname, hdilab01-X299-UD4-Pro resolves to a loopback address: 127.0.1.1; using 172.21.136.90 instead (on interface enp0s31f6)
23/07/19 20:14:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
23/07/19 20:14:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
23/07/19 20:14:07 WARN DependencyUtils: Local jar /home/hdilab01/hdiRecSys/obp_connector/notebooks/jars/replay_2.12-0.1_spark_3.1.jar does not exist, skipping.
23/07/19 20:14:07 INFO SparkContext: Running Spark version 3.1.3
23/07/19 20:14:07 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).
23/07/19 20:14:07 INFO ResourceUtils: ==============================================================
23/

Lets define OpenBanditDataset class with random policy. For the purpose of demonstration we won't use the whole dataset but only subset of size 10000.

In [4]:
data_path= "/home/hdilab01/hdiRecSys/zozo_full/open_bandit_dataset"
behavior_policy = "random"

# Define OBP dataset and split it into train and test
dataset = OpenBanditDataset(behavior_policy=behavior_policy, data_path=data_path, campaign='all')
bandit_feedback_train, bandit_feedback_test = dataset.obtain_batch_bandit_feedback(test_size=0.3, is_timeseries_split=True)

print(bandit_feedback_train["n_rounds"])
print(bandit_feedback_test["n_rounds"])

962028
412299


In [5]:
bandit_feedback_train.keys()

dict_keys(['n_rounds', 'n_actions', 'action', 'position', 'reward', 'pscore', 'context', 'action_context'])

The keys of the dictionary are as follows.
- n_rounds: number of rounds, data size of the logged bandit data;
- n_actions: number of actions $|\mathcal{A}|$;
- action: action variables sampled by the behavior policy;
- position: positions where actions are recommended, there are three positions in the ZOZOTOWN rec interface;
- reward: binary reward variables, click indicators;
- pscore: action choice probabilities by the behavior policy, propensity scores;
- context: context vectors such as user-related features and user-item affinity scores;
- action_context: item-related context vectors

In [6]:
#Define replay model
model = LinUCB(eps=-2.0, alpha=1.0, regr_type="disjoint")

#Define learner which connects OBP data format with replay
learner = RePlayOfflinePolicyLearner(n_actions=dataset.n_actions,
                                     replay_model=model,
                                     len_list=dataset.len_list,) #len_list is the number of predicted items per user

**RePlayOfflinePolicyLearner** has the following methods
- *fit(action, reward, timestamp, context, action_context)*;
- *predict(n_rounds, context)* (context can be None thus n_rounds is **required**);
- *optimize(bandit_feedback, val_size, param_borders, criterion, budget, new_study)*

In [7]:
#Define borders for the optimized parameters
param_borders = {
                "eps": [-10, 10],
                "alpha": [0.001, 10]
}

#Take subset of train data to validate our model with OBP
bandit_feedback_subset = bandit_subset([0, 300000], bandit_feedback_train) #The first parameter is a slice of subset [a, b]
print(learner.optimize(bandit_feedback_subset, val_size=0.3, param_borders=param_borders, budget=30))

[I 2023-07-19 20:14:17,004] A new study created in memory with name: no-name-0b3fe596-89a2-430c-91cf-971103679910
/home/hdilab01/hdiRecSys/ReplayHDILab/replay/optuna_objective.py:76: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use :func:`~optuna.trial.Trial.suggest_float` instead.
  res[param] = suggest_fn(param, low=low, high=high)
/home/hdilab01/anaconda3/envs/rec_sys/lib/python3.9/site-packages/pyspark/sql/pandas/conversion.py:289: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  Unsupported type in conversion from Arrow: uint8
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warnings.warn(msg)
/home/hdilab01/anaconda3/envs/rec_sys/lib/python3.9/site-packages/pyspark/sql/pandas/conversion.py:289: UserWa

[I 2023-07-19 20:17:42,983] Trial 7 finished with value: 0.0008888888888888889 and parameters: {'eps': 5.636457820727125, 'alpha': 0.19076559829832132}. Best is trial 1 with value: 0.005333333333333333.
/home/hdilab01/hdiRecSys/ReplayHDILab/replay/optuna_objective.py:76: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use :func:`~optuna.trial.Trial.suggest_float` instead.
  res[param] = suggest_fn(param, low=low, high=high)
/home/hdilab01/anaconda3/envs/rec_sys/lib/python3.9/site-packages/pyspark/sql/pandas/conversion.py:289: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  Unsupported type in conversion from Arrow: uint8
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warnings.warn(msg)
[I 2023-07-19 20:18:0

/home/hdilab01/anaconda3/envs/rec_sys/lib/python3.9/site-packages/pyspark/sql/pandas/conversion.py:289: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  Unsupported type in conversion from Arrow: uint8
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warnings.warn(msg)
[I 2023-07-19 20:21:25,606] Trial 16 finished with value: 0.0008888888888888889 and parameters: {'eps': 0.2583928443333421, 'alpha': 0.12609841163023164}. Best is trial 1 with value: 0.005333333333333333.
/home/hdilab01/hdiRecSys/ReplayHDILab/replay/optuna_objective.py:76: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use :func:`~optuna.trial.Trial.suggest_float` instead.
  res[param] = suggest_fn(param, low=low, high=high)
/home/hdilab01/anac

[I 2023-07-19 20:24:43,659] Trial 24 finished with value: 0.005333333333333333 and parameters: {'eps': -3.4327632737676037, 'alpha': 3.2844446442029644}. Best is trial 1 with value: 0.005333333333333333.
/home/hdilab01/hdiRecSys/ReplayHDILab/replay/optuna_objective.py:76: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use :func:`~optuna.trial.Trial.suggest_float` instead.
  res[param] = suggest_fn(param, low=low, high=high)
/home/hdilab01/anaconda3/envs/rec_sys/lib/python3.9/site-packages/pyspark/sql/pandas/conversion.py:289: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  Unsupported type in conversion from Arrow: uint8
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warnings.warn(msg)
[I 2023-07-19 20:25:

{'eps': -6.078942275169988, 'alpha': 0.1117365525776511}


In [8]:
#Fit replay model inside our learner
learner.fit(
    action=bandit_feedback_train["action"],
    reward=bandit_feedback_train["reward"],
    timestamp=np.arange(bandit_feedback_train["n_rounds"]),
    context=bandit_feedback_train["context"],
    action_context=bandit_feedback_train["action_context"]
)

#Predict distribution over actions: shape (n_rounds, n_actions, len_list)
action_dist = learner.predict(bandit_feedback_test["n_rounds"], bandit_feedback_test["context"])

print(action_dist.shape)

/home/hdilab01/anaconda3/envs/rec_sys/lib/python3.9/site-packages/pyspark/sql/pandas/conversion.py:289: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  Unsupported type in conversion from Arrow: uint8
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warnings.warn(msg)


(412299, 80, 3)


When we get distribution over actions - we can run any evaluation procedure from the OBP. Here we use three estimators
- *IPW*: Average rewards with importance weights
- *DM*: Average predicted rewards using the classifier
- *DR*: Combination of the above methods with zero bias and lower variance

Also, we can construct confidence intervals for each of these methods.

In [9]:
ope = OffPolicyEvaluation(
    bandit_feedback=bandit_feedback_test,
    ope_estimators=[InverseProbabilityWeighting(), DirectMethod(), DoublyRobust()]
)

estimated_rewards_by_reg_model = get_est_rewards_by_reg(dataset.n_actions,
                                                        dataset.len_list,
                                                        bandit_feedback_train,
                                                        bandit_feedback_test)

estimated_policy_value = ope.estimate_policy_values(
    action_dist=action_dist,
    estimated_rewards_by_reg_model=estimated_rewards_by_reg_model,
)

estimated_ci = ope.estimate_intervals(
    action_dist=action_dist,
    estimated_rewards_by_reg_model=estimated_rewards_by_reg_model,
    n_bootstrap_samples=10000,
    random_state=12345,)

In [10]:
out_str = f"Scores for LinUCB"
for key, val in estimated_policy_value.items():
    out_str += f" {key} : {(1e3 * val):.3f},"

out_str = out_str[:-1]

print(out_str)
print("Estimated confidence intervals:")
print(pd.DataFrame(estimated_ci).to_string())

Scores for LinUCB ipw : 4.463, dm : 3.539, dr : 4.409
Estimated confidence intervals:
                       ipw        dm        dr
mean              0.004440  0.003539  0.004386
95.0% CI (lower)  0.002716  0.003535  0.002657
95.0% CI (upper)  0.006403  0.003542  0.006306
